In [1]:
# import libraries
import polars as pl
from datetime import date
from tqdm.notebook import tqdm
import plotly.express as px

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_width_chars(200)

# data path
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent.parent))

from src.config.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DOCS_DIR

In [2]:
# load cleaned data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# 8. Data Model Preparation

**Main Objective**

Transform the selected origination-time features into a model-ready representation while preserving predictive information, interpretability, and leakage control.

## 8.1 One Hot Encoding

In [3]:
CAT_FEATURES = [i for i,d in df.collect_schema().items() if not d.is_numeric()]
CAT_FEATURES.remove("issue_d")

CAT_FEATURE_VALUES = {}

for feature in CAT_FEATURES:
    values = (
        df
        .select(
            pl.col(feature)
            .drop_nulls()
            .cast(pl.String)
            .unique()
            .sort()
        )
        .collect()
        .to_series()
        .to_list()
    )

    CAT_FEATURE_VALUES[feature] = values

In [4]:
print(CAT_FEATURE_VALUES)

{'grade': ['A', 'B', 'C', 'D', 'E', 'F', 'G'], 'initial_list_status': ['f', 'w'], 'application_type': ['Individual', 'Joint App'], 'purpose': ['car', 'credit_card', 'debt_consolidation', 'educational', 'home_improvement', 'house', 'major_purchase', 'medical', 'moving', 'other', 'renewable_energy', 'small_business', 'vacation', 'wedding'], 'home_ownership': ['ANY', 'MORTGAGE', 'NONE', 'OWN', 'RENT'], 'verification_status': ['Not Verified', 'Source Verified', 'Verified'], 'addr_state': ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']}


In [5]:
from src.config.categories import OHE_CATEGORIES

In [6]:
def one_hot_encode(df: pl.DataFrame | pl.LazyFrame,):
    expressions = []

    for feature, categories in OHE_CATEGORIES.items():

        for category in categories:

            encoded_name = f"{feature}_{category}"

            expressions.append(
                (
                    pl.col(feature).cast(pl.String)
                    == pl.lit(category)
                )
                .fill_null(False)
                .cast(pl.UInt8)
                .alias(encoded_name)
            )

    return (df
            .with_columns(expressions)
            .select(pl.exclude(OHE_CATEGORIES.keys())))

In [7]:
df = one_hot_encode(df)

In [8]:
df.head().sort("issue_d").collect()

loan_amnt,term,int_rate,emp_length,annual_inc,fico_range_low,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,num_tl_op_past_12m,dti,total_acc,open_act_il,open_il_12m,open_rv_24m,all_util,total_cu_tl,pct_tl_nvr_dlq,mort_acc,tot_cur_bal,num_actv_rev_tl,revol_util,max_bal_bc,mths_since_recent_bc,total_bc_limit,mths_since_rcnt_il,total_bal_il,il_util,inq_last_6mths,inq_fi,inq_last_12m,delinq_2yrs,num_accts_ever_120_pd,num_tl_120dpd_2m,num_tl_90g_dpd_24m,collections_12_mths_ex_med,pub_rec,pub_rec_bankruptcies,tot_coll_amt,num_il_tl,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null,emp_length_null,cr_age_mths,default,issue_d,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,initial_list_status_f,initial_list_status_w,application_type_Individual,application_type_Joint App,purpose_car,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,addr_state_AK,addr_state_AL,addr_state_AR,addr_state_AZ,addr_state_CA,addr_state_CO,addr_state_CT,addr_state_DC,addr_state_DE,addr_state_FL,addr_state_GA,addr_state_HI,addr_state_ID,addr_state_IL,addr_state_IN,addr_state_KS,addr_state_KY,addr_state_LA,addr_state_MA,addr_state_MD,addr_state_ME,addr_state_MI,addr_state_MN,addr_state_MO,addr_state_MS,addr_state_MT,addr_state_NC,addr_state_ND,addr_state_NE,addr_state_NH,addr_state_NJ,addr_state_NM,addr_state_NV,addr_state_NY,addr_state_OH,addr_state_OK,addr_state_OR,addr_state_PA,addr_state_RI,addr_state_SC,addr_state_SD,addr_state_TN,addr_state_TX,addr_state_UT,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY
i64,i64,f64,i64,f64,i64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,i8,i8,i8,i8,i64,i8,date,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
12000,36,7.97,0,42000.0,715,131.0,255,1,1,3,27.74,16,2.0,1.0,4.0,53.0,1.0,100.0,0,30502.0,6,37.0,7117.0,14,15500.0,8,19045.0,73.0,0.0,1.0,2.0,0,0,0.0,0,0,1,1,0.0,7,0,0,1,1,258,0,2017-09-01,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10000,36,9.44,3,55000.0,695,144.0,73,49,10,1,18.79,10,4.0,1.0,0.0,68.0,1.0,100.0,2,340607.0,2,57.5,6847.0,49,10500.0,10,209187.0,75.0,0.0,1.0,1.0,0,0,0.0,0,0,1,1,0.0,5,0,0,1,0,146,0,2017-09-01,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8000,36,16.02,0,120000.0,700,137.0,276,34,6,3,20.36,34,3.0,2.0,0.0,88.0,24.0,97.1,6,388595.0,3,92.3,0.0,152,0.0,6,83572.0,85.0,1.0,1.0,2.0,0,0,0.0,0,0,0,0,0.0,19,0,1,1,0,280,0,2017-09-01,0,0,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
12800,36,13.59,5,90000.0,660,154.0,345,5,5,3,22.63,23,3.0,2.0,3.0,86.0,1.0,83.0,0,93375.0,6,86.0,3777.0,5,14750.0,6,80715.0,91.0,2.0,0.0,2.0,0,0,0.0,0,0,0,0,0.0,15,0,1,0,0,350,0,2017-09-01,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
15000,36,13.59,4,180000.0,680,132.0,489,16,13,0,38.07,50,4.0,0.0,1.0,68.0,8.0,100.0,4,682000.0,14,66.6,22127.0,16,116700.0,13

## 8.2 Temporal Split

In [9]:
total_rows = df.select(pl.len()).collect().item()

issue_d_cumsum = (df
                    .select("issue_d")
                    .group_by("issue_d")
                    .agg(pl.len().alias("count"))
                    .sort("issue_d")
                    .with_columns(
                        (pl.col("count") / total_rows * 100.0).cast(pl.Float64).round(2).alias("pct"),
                        (pl.col("count").cum_sum().alias("count_cumsum"))
                    )
                    .with_columns(
                        (pl.col("count_cumsum") / total_rows * 100.0).cast(pl.Float64).round(2).alias("pct_cumsum")
                    )
                    .collect())

In [10]:
import plotly.express as px
import pandas as pd

df_pd = issue_d_cumsum.to_pandas()

# Convert the first and last dates to strings with format YYYY-MM-DD
first_date_str = df_pd['issue_d'].min().strftime('%Y-%m-%d')
last_date_str = df_pd['issue_d'].max().strftime('%Y-%m-%d')

fig = px.line(
    df_pd,
    x="issue_d",
    y="pct_cumsum",
    markers=True,
    title="Data Cumulative Split (85 / 10 / 5)",
    labels={
        "issue_d": "Vintage",
        "pct_cumsum": "Percentage",
    },
)

# 1. Highlight Train Area (Start to Jul 2018)
fig.add_vrect(
    x0=first_date_str, 
    x1="2018-07-01",
    fillcolor="green", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Train (85%)", 
    annotation_position="top left"
)

# 2. Highlight Validation Area (Aug 2018 to Mar 2019)
fig.add_vrect(
    x0="2018-07-01", 
    x1="2019-03-01",
    fillcolor="orange", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Val (10%)", 
    annotation_position="top left"
)

# 3. Highlight Test Area (Apr 2019 to End)
fig.add_vrect(
    x0="2019-03-01", 
    x1=last_date_str,
    fillcolor="red", 
    opacity=0.1, 
    layer="below", 
    line_width=0,
    annotation_text="Test (5%)", 
    annotation_position="top left"
)

fig.show()

In [12]:
from src.data.split import temporal_split, split_xy

df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

In [15]:
df_train.head().collect()

loan_amnt,term,int_rate,emp_length,annual_inc,fico_range_low,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,num_tl_op_past_12m,dti,total_acc,open_act_il,open_il_12m,open_rv_24m,all_util,total_cu_tl,pct_tl_nvr_dlq,mort_acc,tot_cur_bal,num_actv_rev_tl,revol_util,max_bal_bc,mths_since_recent_bc,total_bc_limit,mths_since_rcnt_il,total_bal_il,il_util,inq_last_6mths,inq_fi,inq_last_12m,delinq_2yrs,num_accts_ever_120_pd,num_tl_120dpd_2m,num_tl_90g_dpd_24m,collections_12_mths_ex_med,pub_rec,pub_rec_bankruptcies,tot_coll_amt,num_il_tl,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null,emp_length_null,cr_age_mths,default,issue_d,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,initial_list_status_f,initial_list_status_w,application_type_Individual,application_type_Joint App,purpose_car,purpose_credit_card,purpose_debt_consolidation,purpose_educational,purpose_home_improvement,purpose_house,purpose_major_purchase,purpose_medical,purpose_moving,purpose_other,purpose_renewable_energy,purpose_small_business,purpose_vacation,purpose_wedding,home_ownership_ANY,home_ownership_MORTGAGE,home_ownership_NONE,home_ownership_OWN,home_ownership_RENT,verification_status_Not Verified,verification_status_Source Verified,verification_status_Verified,addr_state_AK,addr_state_AL,addr_state_AR,addr_state_AZ,addr_state_CA,addr_state_CO,addr_state_CT,addr_state_DC,addr_state_DE,addr_state_FL,addr_state_GA,addr_state_HI,addr_state_ID,addr_state_IL,addr_state_IN,addr_state_KS,addr_state_KY,addr_state_LA,addr_state_MA,addr_state_MD,addr_state_ME,addr_state_MI,addr_state_MN,addr_state_MO,addr_state_MS,addr_state_MT,addr_state_NC,addr_state_ND,addr_state_NE,addr_state_NH,addr_state_NJ,addr_state_NM,addr_state_NV,addr_state_NY,addr_state_OH,addr_state_OK,addr_state_OR,addr_state_PA,addr_state_RI,addr_state_SC,addr_state_SD,addr_state_TN,addr_state_TX,addr_state_UT,addr_state_VA,addr_state_VT,addr_state_WA,addr_state_WI,addr_state_WV,addr_state_WY
i64,i64,f64,i64,f64,i64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,i8,i8,i8,i8,i64,i8,date,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8
12000,36,7.97,0,42000.0,715,131.0,255,1,1,3,27.74,16,2.0,1.0,4.0,53.0,1.0,100.0,0,30502.0,6,37.0,7117.0,14,15500.0,8,19045.0,73.0,0.0,1.0,2.0,0,0,0.0,0,0,1,1,0.0,7,0,0,1,1,258,0,2017-09-01,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
10000,36,9.44,3,55000.0,695,144.0,73,49,10,1,18.79,10,4.0,1.0,0.0,68.0,1.0,100.0,2,340607.0,2,57.5,6847.0,49,10500.0,10,209187.0,75.0,0.0,1.0,1.0,0,0,0.0,0,0,1,1,0.0,5,0,0,1,0,146,0,2017-09-01,0,1,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
8000,36,16.02,0,120000.0,700,137.0,276,34,6,3,20.36,34,3.0,2.0,0.0,88.0,24.0,97.1,6,388595.0,3,92.3,0.0,152,0.0,6,83572.0,85.0,1.0,1.0,2.0,0,0,0.0,0,0,0,0,0.0,19,0,1,1,0,280,0,2017-09-01,0,0,1,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
12800,36,13.59,5,90000.0,660,154.0,345,5,5,3,22.63,23,3.0,2.0,3.0,86.0,1.0,83.0,0,93375.0,6,86.0,3777.0,5,14750.0,6,80715.0,91.0,2.0,0.0,2.0,0,0,0.0,0,0,0,0,0.0,15,0,1,0,0,350,0,2017-09-01,0,0,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
15000,36,13.59,4,180000.0,680,132.0,489,16,13,0,38.07,50,4.0,0.0,1.0,68.0,8.0,100.0,4,682000.0,14,66.6,22127.0,16,116700.0,13

## 8.3 Weight of Evidence (WoE)

In [18]:
from src.data.split import temporal_split, split_xy
from src.data.woe import WoEEncoder

# load cleaned data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# temporal split
df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

# woe
woe      = WoEEncoder(min_samples=2)
df_train = woe.fit_transform(df_train)
df_val   = woe.transform(df_val)
df_test  = woe.transform(df_test)

# check iv summary before proceeding
print(woe.iv_summary())

# drop weak features across all splits
weak_cols = woe.drop_weak_features(threshold=0.02)
if weak_cols:
    print(f"dropping {len(weak_cols)} weak features: {weak_cols}")
    df_train = df_train.drop(weak_cols)
    df_val   = df_val.drop(weak_cols)
    df_test  = df_test.drop(weak_cols)

woe encoder saved to /home/adhitizki/Project/202607_credit-risk-personal/artifact/preprocessing/woe_encoder.pkl
               feature        iv predictive_power
0                grade  0.453791           strong
1  verification_status  0.053164             weak
2       home_ownership  0.034786             weak
3           addr_state  0.017355          useless
4              purpose  0.015975          useless
5     application_type  0.004744          useless
6  initial_list_status  0.000705          useless
dropping 4 weak features: ['initial_list_status_woe', 'application_type_woe', 'purpose_woe', 'addr_state_woe']


In [20]:
df_train.collect()

loan_amnt,term,int_rate,emp_length,annual_inc,fico_range_low,mo_sin_old_il_acct,mo_sin_old_rev_tl_op,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,num_tl_op_past_12m,dti,total_acc,open_act_il,open_il_12m,open_rv_24m,all_util,total_cu_tl,pct_tl_nvr_dlq,mort_acc,tot_cur_bal,num_actv_rev_tl,revol_util,max_bal_bc,mths_since_recent_bc,total_bc_limit,mths_since_rcnt_il,total_bal_il,il_util,inq_last_6mths,inq_fi,inq_last_12m,delinq_2yrs,num_accts_ever_120_pd,num_tl_120dpd_2m,num_tl_90g_dpd_24m,collections_12_mths_ex_med,pub_rec,pub_rec_bankruptcies,tot_coll_amt,num_il_tl,mths_since_recent_inq_null,mths_since_last_record_null,mths_since_last_major_derog_null,emp_length_null,cr_age_mths,default,issue_d,grade_woe,home_ownership_woe,verification_status_woe
i64,i64,f64,i64,f64,i64,f64,i64,i64,i64,i64,f64,i64,f64,f64,f64,f64,f64,f64,i64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,f64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,i8,i8,i8,i8,i64,i8,datetime[ms],f64,f64,f64
12000,36,7.97,0,42000.0,715,131.0,255,1,1,3,27.74,16,2.0,1.0,4.0,53.0,1.0,100.0,0,30502.0,6,37.0,7117.0,14,15500.0,8,19045.0,73.0,0.0,1.0,2.0,0,0,0.0,0,0,1,1,0.0,7,0,0,1,1,258,0,2017-09-01 00:00:00,-1.292049,0.011345,0.010747
10000,36,9.44,3,55000.0,695,144.0,73,49,10,1,18.79,10,4.0,1.0,0.0,68.0,1.0,100.0,2,340607.0,2,57.5,6847.0,49,10500.0,10,209187.0,75.0,0.0,1.0,1.0,0,0,0.0,0,0,1,1,0.0,5,0,0,1,0,146,0,2017-09-01 00:00:00,-0.425884,-0.191622,-0.308441
8000,36,16.02,0,120000.0,700,137.0,276,34,6,3,20.36,34,3.0,2.0,0.0,88.0,24.0,97.1,6,388595.0,3,92.3,0.0,152,0.0,6,83572.0,85.0,1.0,1.0,2.0,0,0,0.0,0,0,0,0,0.0,19,0,1,1,0,280,0,2017-09-01 00:00:00,0.20202,-0.191622,-0.308441
12800,36,13.59,5,90000.0,660,154.0,345,5,5,3,22.63,23,3.0,2.0,3.0,86.0,1.0,83.0,0,93375.0,6,86.0,3777.0,5,14750.0,6,80715.0,91.0,2.0,0.0,2.0,0,0,0.0,0,0,0,0,0.0,15,0,1,0,0,350,0,2017-09-01 00:00:00,0.20202,0.206879,-0.308441
15000,36,13.59,4,180000.0,680,132.0,489,16,13,0,38.07,50,4.0,0.0,1.0,68.0,8.0,100.0,4,682000.0,14,66.6,22127.0,16,116700.0,13,190233.0,79.0,0.0,2.0,0.0,0,0,0.0,0,0,0,0,0.0,17,0,1,1,0,496,0,2017-09-01 00:00:00,0.20202,-0.191622,0.010747
40000,60,7.97,0,200000.0,760,155.0,157,0,0,2,14.37,68,9.0,1.0,1.0,54.0,8.0,100.0,6,354714.0,6,13.1,2343.0,28,48400.0,11,344236.0,99.0,0.0,4.0,7.0,0,0,0.0,0,0,0,0,0.0,35,0,1,1,1,159,1,2017-09-01 00:00:00,-1.292049,0.011345,0.294062
18000,60,16.02,0,125000.0,675,132.0,124,5,5,4,9.53,31,2.0,0.0,6.0,63.0,0.0,54.8,0,57941.0,11,34.9,5629.0,5,51600.0,30,39958.0,100.0,1.0,0.0,2.0,2,10,0.0,2,0,0,0,0.0,18,0,1,0,0,133,0,2017-09-01 00:00:00,0.20202,0.206879,-0.308441
2000,36,23.88,0,24000.0,660,190.0,169,13,13,0,42.2,16,2.0,0.0,2.0,65.0,0.0,93.3,1,21836.0,4,70.1,2226.0,40,5800.0,15,14615.0,62.0,0.0,1.0,2.0,0,1,0.0,0,0,0,0,165.0,8,0,1,0,1,192,0,2017-09-01 00:00:00,0.986474,0.206879,0.294062
16000,60,17.09,0,60000.0,660,10.0,169,0,0,4,24.72,33,1.0,1.0,6.0,85.0,0.0,73.0,0,42983.0,13,85.0,7259.0,13,30600.0,10,11888.0,93.0,4.0,1.0,4.0,0,0,0.0,0,0,1,1,0.0,1,0,0,0,0,171,1,2017-09-01 00:00:00,0.642572,0.206879,0.010747


## 8.4 Data Transformation for Model Ready

In [22]:
import polars as pl

from src.data.ohe import one_hot_encode
from src.data.woe import WoEEncoder
from src.data.scaling import FeatureScaler
from src.data.split import temporal_split, split_xy

In [36]:
def split_save_data(df_train, df_val, df_test, 
                    init_name,
                    scaling=False):

    # split X y
    X_train, y_train = split_xy(df_train)
    X_val, y_val = split_xy(df_val)
    X_test, y_test = split_xy(df_test)

    # scaling
    if scaling:
        scaler = FeatureScaler()

        X_train = scaler.fit_transform(X_train)
        X_val   = scaler.transform(X_val)
        X_test  = scaler.transform(X_test)

        init_name = f"{init_name}_scale"

    # make directory
    dir_name = PROCESSED_DIR / f"{init_name}"
    Path(dir_name).mkdir(parents=True, exist_ok=True)

    # save file
    X_train.sink_parquet(dir_name / "X_train.parquet")
    y_train.sink_parquet(dir_name / "y_train.parquet")
    X_val.sink_parquet(dir_name / "X_val.parquet")
    y_val.sink_parquet(dir_name / "y_val.parquet")
    X_test.sink_parquet(dir_name / "X_test.parquet")
    y_test.sink_parquet(dir_name / "y_test.parquet")


### 8.4.1 OHE

In [38]:
# load data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# ohe
df = one_hot_encode(df)

# split
df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

# split Xy and save
split_save_data(df_train, df_val, df_test, "ohe")

### 8.4.2 WoE

In [40]:
# load data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# split
df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

# woe
woe      = WoEEncoder(min_samples=2)
df_train = woe.fit_transform(df_train)  # fits on train target, transforms
df_val   = woe.transform(df_val)        # applies train woe maps
df_test  = woe.transform(df_test)       # applies train woe maps

# check iv summary before proceeding
print(woe.iv_summary())

# drop weak features across all splits
weak_cols = woe.drop_weak_features(threshold=0.02)
if weak_cols:
    print(f"dropping {len(weak_cols)} weak features: {weak_cols}")
    df_train = df_train.drop(weak_cols)
    df_val   = df_val.drop(weak_cols)
    df_test  = df_test.drop(weak_cols)

# split Xy and save
split_save_data(df_train, df_val, df_test, "woe")

woe encoder saved to /home/adhitizki/Project/202607_credit-risk-personal/artifact/preprocessing/woe_encoder.pkl
               feature        iv predictive_power
0                grade  0.453791           strong
1  verification_status  0.053164             weak
2       home_ownership  0.034786             weak
3           addr_state  0.017355          useless
4              purpose  0.015975          useless
5     application_type  0.004744          useless
6  initial_list_status  0.000705          useless
dropping 4 weak features: ['initial_list_status_woe', 'application_type_woe', 'purpose_woe', 'addr_state_woe']


### 8.4.3 OHE + Scaling

In [41]:
# load data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# ohe
df = one_hot_encode(df)

# split
df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

# split Xy and save
split_save_data(df_train, df_val, df_test, "ohe", True)

scaler saved to /home/adhitizki/Project/202607_credit-risk-personal/artifact/preprocessing/scaler.pkl


### 8.4.4 WoE + Scaling

In [42]:
# load data
df = pl.scan_parquet(INTERIM_DIR / "cleaned.parquet")

# split
df_train, df_val, df_test = temporal_split(df, val_start="2018-08-01", test_start="2019-04-01")

# woe
woe      = WoEEncoder(min_samples=2)
df_train = woe.fit_transform(df_train)  # fits on train target, transforms
df_val   = woe.transform(df_val)        # applies train woe maps
df_test  = woe.transform(df_test)       # applies train woe maps

# check iv summary before proceeding
print(woe.iv_summary())

# drop weak features across all splits
weak_cols = woe.drop_weak_features(threshold=0.02)
if weak_cols:
    print(f"dropping {len(weak_cols)} weak features: {weak_cols}")
    df_train = df_train.drop(weak_cols)
    df_val   = df_val.drop(weak_cols)
    df_test  = df_test.drop(weak_cols)

# split Xy and save
split_save_data(df_train, df_val, df_test, "woe", True)

woe encoder saved to /home/adhitizki/Project/202607_credit-risk-personal/artifact/preprocessing/woe_encoder.pkl
               feature        iv predictive_power
0                grade  0.453791           strong
1  verification_status  0.053164             weak
2       home_ownership  0.034786             weak
3           addr_state  0.017355          useless
4              purpose  0.015975          useless
5     application_type  0.004744          useless
6  initial_list_status  0.000705          useless
dropping 4 weak features: ['initial_list_status_woe', 'application_type_woe', 'purpose_woe', 'addr_state_woe']
scaler saved to /home/adhitizki/Project/202607_credit-risk-personal/artifact/preprocessing/scaler.pkl
